In [26]:
import pandas as pd
import numpy as np
import os

DATA_DIR = "../data/raw/"
PROCESSED_DIR = "../data/processed/"

train_df = pd.read_csv(DATA_DIR + "train.csv")
test_df  = pd.read_csv(DATA_DIR + "test.csv")


y_train = train_df["Survived"].copy()
train_df = train_df.drop(columns=["Survived"])

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("Target shape:", y_train.shape)

train_df.head()

Train shape: (891, 11)
Test shape : (418, 11)
Target shape: (891,)


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [27]:
for df in [train_df, test_df]:
    df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.")


print("=== Train Titles (before grouping) ===")
print(train_df["Title"].value_counts())

=== Train Titles (before grouping) ===
Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64


In [28]:
rare_map= {
    "Mile":"Miss",
    "Ms":"Miss",
    "Mme":"Mrs"
}

for df in [train_df, test_df]:
    df['Title']=df['Title'].replace(rare_map)
    df["Title"]= df["Title"].where(
        df["Title"].isin(["Mr","Mrs","Miss","Master"]),
        "Rare"
    )

print("=== Train Titles(after consolidation) ===")
print(train_df["Title"].value_counts())

print("\n=== Test Titles (after consolidation) ===")
print(test_df["Title"].value_counts())

=== Train Titles(after consolidation) ===
Title
Mr        517
Miss      183
Mrs       126
Master     40
Rare       25
Name: count, dtype: int64

=== Test Titles (after consolidation) ===
Title
Mr        240
Miss       79
Mrs        72
Master     21
Rare        6
Name: count, dtype: int64


In [29]:
for df in [train_df, test_df]:
    df['FamilySize']= df['SibSp'] + df['Parch'] + 1
    df['IsAlone']= (df['FamilySize']==1).astype(int)
    
print("=== FamilySize Distribution (Train) ===")
print(train_df["FamilySize"].value_counts().sort_index())

print("\n=== IsAlone Counts (Train) ===")
print(train_df["IsAlone"].value_counts())

print("\n=== Survival Rate by FamilySize (Train) ===")
print(train_df.assign(Survived=y_train).groupby("FamilySize")["Survived"].mean().round(4))



=== FamilySize Distribution (Train) ===
FamilySize
1     537
2     161
3     102
4      29
5      15
6      22
7      12
8       6
11      7
Name: count, dtype: int64

=== IsAlone Counts (Train) ===
IsAlone
1    537
0    354
Name: count, dtype: int64

=== Survival Rate by FamilySize (Train) ===
FamilySize
1     0.3035
2     0.5528
3     0.5784
4     0.7241
5     0.2000
6     0.1364
7     0.3333
8     0.0000
11    0.0000
Name: Survived, dtype: float64


In [30]:
for df in [train_df, test_df]:
    df['HasCabin'] = df['Cabin'].notnull().astype(int)

print("=== HasCabin Counts (Train) ===")
print(train_df["HasCabin"].value_counts())

print(train_df.assign(Survived= y_train).groupby("HasCabin")["Survived"].mean().round(4))

=== HasCabin Counts (Train) ===
HasCabin
0    687
1    204
Name: count, dtype: int64
HasCabin
0    0.2999
1    0.6667
Name: Survived, dtype: float64


In [32]:

print("=== Missing Ages BEFORE imputation ===")
print(f"Train: {train_df['Age'].isnull().sum()}")
print(f"Test : {test_df['Age'].isnull().sum()}")

for df in [train_df, test_df]:
    df["Age"] = df.groupby("Title")["Age"].transform(
        lambda x: x.fillna(x.median())
    )

print("\n=== Missing Ages AFTER imputation ===")
print(f"Train: {train_df['Age'].isnull().sum()}")
print(f"Test : {test_df['Age'].isnull().sum()}")

=== Missing Ages BEFORE imputation ===
Train: 0
Test : 0

=== Missing Ages AFTER imputation ===
Train: 0
Test : 0


In [37]:
# Block 7: Impute Embarked + drop useless columns
print("=== Missing Embarked BEFORE ===")
print(train_df["Embarked"].isnull().sum())

for df in [train_df, test_df]:
    df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

print("\n=== Missing Embarked AFTER ===")
print(train_df["Embarked"].isnull().sum())

# Drop columns the model can't use — errors="ignore" makes it re-run safe
drop_cols = ["PassengerId", "Name", "Ticket", "Cabin"]
for df in [train_df, test_df]:
    df.drop(columns=drop_cols, inplace=True, errors="ignore")

print("\n=== Remaining Columns (Train) ===")
print(train_df.columns.tolist())
print("\nShape:", train_df.shape)

=== Missing Embarked BEFORE ===
0

=== Missing Embarked AFTER ===
0

=== Remaining Columns (Train) ===
['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone', 'HasCabin']

Shape: (891, 11)


In [38]:
# Block 8: Save cleaned data + final sanity check
import os

os.makedirs(PROCESSED_DIR, exist_ok=True)

# Reattach target to train for saving
train_clean = train_df.copy()
train_clean["Survived"] = y_train.values

train_clean.to_csv(PROCESSED_DIR + "train_clean.csv", index=False)
test_df.to_csv(PROCESSED_DIR + "test_clean.csv", index=False)

print("=== Saved ===")
print(f"train_clean.csv → {train_clean.shape}")
print(f"test_clean.csv  → {test_df.shape}")

print("\n=== Final Column Check ===")
print("Train columns:", train_clean.columns.tolist())
print("Test columns :", test_df.columns.tolist())

print("\n=== Remaining Missing Values ===")
print("Train:", train_clean.isnull().sum().sum())
print("Test :", test_df.isnull().sum().sum())

print("\n=== Preview ===")
print(train_clean.head())

=== Saved ===
train_clean.csv → (891, 12)
test_clean.csv  → (418, 11)

=== Final Column Check ===
Train columns: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone', 'HasCabin', 'Survived']
Test columns : ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone', 'HasCabin']

=== Remaining Missing Values ===
Train: 0
Test : 1

=== Preview ===
   Pclass     Sex   Age  SibSp  Parch     Fare Embarked Title  FamilySize  \
0       3    male  22.0      1      0   7.2500        S    Mr           2   
1       1  female  38.0      1      0  71.2833        C   Mrs           2   
2       3  female  26.0      0      0   7.9250        S  Miss           1   
3       1  female  35.0      1      0  53.1000        S   Mrs           2   
4       3    male  35.0      0      0   8.0500        S    Mr           1   

   IsAlone  HasCabin  Survived  
0        0         0         0  
1        0         1         1  
2        1 